# Teks — Topic Modeling dengan BERTopic (embedding kalimat + clustering)
**Kapan dipakai:** ingin topik yang lebih "bermakna" daripada LDA/NMF (`text/topik`),
terutama untuk teks pendek. Jumlah topik tidak wajib ditentukan di awal.

**Cara kerja:**
1. Embedding kalimat (MiniLM lokal) → tiap dokumen jadi vektor angka.
2. UMAP mengecilkan dimensi, HDBSCAN mengelompokkan.
3. Kata khas tiap kelompok dicari dengan c-TF-IDF.

Dokumen yang tidak masuk kelompok mana pun = topik -1.

**Data demo:** news4 (4 topik). Label HANYA dipakai untuk membandingkan (NMI/ARI).

In [ ]:
import pandas as pd
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from umap import UMAP

## Konfigurasi

In [ ]:
# ============ NILAI SOAL: isi SAMA di semua notebook 1_..7_ (tidak ada konfig.py) ============
# Sesudah mengubah: Restart Kernel + Run All. Nilai awal = data demo SmSA (sentimen, 3 kelas).
# ---- File soal (garis miring /, bukan \) ----
# ---- Kolom ----
TEXT_COLS = ["text"]                          # GANTI: kolom teks. 4_finetune hanya memakai yang pertama
TARGET = "label"                              # GANTI: kolom kelas
SUBMIT_COL = TARGET                           # GANTI bila kolom prediksi di sample bernama lain
LABEL_COLS = None                             # MULTILABEL: daftar kolom label 0/1, mis. ["a", "b"].
#                                               None = single label. Bila diisi: pakai 3m, 4m, 5m
LANGUAGE = "id"                               # "id" = stopword Indonesia, "en" = Inggris
# ---- Hasil ----
HASIL = "../../outputs/final"                 # GANTI NAMA bila ganti dataset (mis. "../../outputs/lomba"),
#                                               supaya OOF demo tidak tercampur dengan OOF soal

# Coba jalur multilabel di data demo CASA (ulasan mobil, 6 label): buka # di empat baris ini
# TRAIN_MENTAH = "demo/casa_train.csv"
# TEST_MENTAH = "demo/casa_test.csv"
# LABEL_COLS = ["fuel", "machine", "others", "part", "price", "service"]
# HASIL = "../../outputs/final_multilabel"
# Multilabel: EDA, praproses, dan "topik x kelas" butuh SATU kolom kelas. Bila TARGET bukan salah satu
# LABEL_COLS, dipakai label pertama (ganti TARGET ke label yang paling penting).
if LABEL_COLS is not None and TARGET not in LABEL_COLS:
    TARGET = LABEL_COLS[0]
    SUBMIT_COL = TARGET
# ====================================================================================================

# teks Indonesia: EMBEDDING_MODEL = "../../pretrained/minilm"

MIN_TOPIC_SIZE = 50       # kelompok lebih kecil dari ini digabung / jadi -1
SEED = 42

# ---- jalur final-pipeline: dihitung dari HASIL di atas, tidak perlu diubah ----
DATA_CSV = f"{HASIL}/praproses/train_bersih.csv"
TEXT_COL = TEXT_COLS[0]
LABEL_COL = TARGET
EMBEDDING_MODEL = "../../pretrained/minilm"
N_TOPICS = None           # None = otomatis (HDBSCAN), atau angka = target topik + 1
OUTPUT_FOLDER = f"{HASIL}/bertopic"
import os
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

## Baca data

In [ ]:
data = pd.read_csv(DATA_CSV)
data[TEXT_COL] = data[TEXT_COL].fillna("")
texts = list(data[TEXT_COL])
print(data.shape)
print(data[LABEL_COL].value_counts())

## Embedding kalimat
Dihitung sekali; saat mencoba pengaturan BERTopic lain, sel ini tidak perlu dijalankan ulang.

In [ ]:
encoder = SentenceTransformer(EMBEDDING_MODEL)
embeddings = encoder.encode(texts, batch_size=64, show_progress_bar=False)
print(embeddings.shape)

## BERTopic
`random_state` di UMAP supaya hasil bisa diulang. Stopword dibuang hanya saat memilih kata
topik.

In [ ]:
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine",
                  random_state=SEED)
hdbscan_model = HDBSCAN(min_cluster_size=MIN_TOPIC_SIZE, metric="euclidean",
                        prediction_data=True)
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
stop_words = None
if LANGUAGE == "id":
    stop_words = StopWordRemoverFactory().get_stop_words()
if LANGUAGE == "en":
    stop_words = "english"
word_counter = CountVectorizer(stop_words=stop_words, min_df=2)
# teks Indonesia: word_counter = CountVectorizer(stop_words=daftar_stopword_id, min_df=2)

model = BERTopic(
    embedding_model=encoder,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=word_counter,
    nr_topics=N_TOPICS,
)
topics, probabilities = model.fit_transform(texts, embeddings)
topic_info = model.get_topic_info()
topic_info.head(10)

## Kata kunci tiap topik

In [ ]:
for topic_number in topic_info["Topic"]:
    if topic_number == -1:
        continue
    words = []
    for word, weight in model.get_topic(topic_number)[:10]:
        words.append(word)
    print("topik", topic_number, ":", ", ".join(words))

## Evaluasi dengan label (HANYA DEMO; saat lomba biasanya tidak ada label)
Pembanding: LDA di `text/topik` (NMI 0.41, ARI 0.39).

In [ ]:
data["topik"] = topics
print("dokumen tanpa topik (-1):", round((data["topik"] == -1).mean(), 3))
print("NMI:", round(normalized_mutual_info_score(data[LABEL_COL], data["topik"]), 3))
print("ARI:", round(adjusted_rand_score(data[LABEL_COL], data["topik"]), 3))
print(pd.crosstab(data[LABEL_COL], data["topik"]))

## Simpan topik tiap dokumen + info topik

In [ ]:
result = pd.DataFrame()
result["text"] = data[TEXT_COL]
result["topik"] = data["topik"]
result.to_csv(f"{OUTPUT_FOLDER}/topik_dokumen.csv", index=False)   # per baris: JANGAN dibawa
topic_info.drop(columns=["Representative_Docs"]).to_csv(f"{OUTPUT_FOLDER}/info_topik.csv",
                                                        index=False)   # tanpa teks asli
result.head()